<a href="https://colab.research.google.com/github/JoseORHub/Alertas_Invima/blob/main/Scrapping_Alertas_Invima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# INVIMA - Scraper de alertas de medicamentos
# ============================================================
#   1. Ejecuta la celda de instalación primero
# ============================================================

!pip install -q pandas openpyxl beautifulsoup4 lxml pymupdf requests pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 11.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 2. Luego ejecuta este script
# ============================================================

import re
import io
import time
from datetime import datetime
from urllib.parse import urljoin

import requests
from pypdf import PdfReader
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIGURACIÓN
# ============================================================

BASE_URL = "https://app.invima.gov.co/alertas/medicamentos-productos-biologicos"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "es-CO,es;q=0.9,en;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Referer": BASE_URL,
}

REQUEST_TIMEOUT = 60
PDF_TIMEOUT = 90
PAUSE_BETWEEN_PAGES = 0.6
PAUSE_BETWEEN_PDFS = 0.1
MAX_RETRIES = 3
RETRY_WAIT = 2

SPANISH_MONTHS = (
    r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|"
    r"septiembre|setiembre|octubre|noviembre|diciembre)"
)
RE_DATE   = re.compile(rf"(?i)\bBogotá,?\s*\d{{1,2}}\s+(?:de\s+)?{SPANISH_MONTHS}\s+(?:de\s+)?\d{{4}}\b")
RE_ALERT  = re.compile(r"(?i)\b(Alerta|Informe\s+de\s+Seguridad)\s+No\.?\s*#?\s*([A-Za-z0-9\-]+)")
RE_RISARH = re.compile(r"\b(MA\d{3,}-\d+|\d{4}-\d{4}-\d{4})\b", re.IGNORECASE)
RE_SKIP   = re.compile(r"(?i)alerta\s+no|bogotá|invima|subdirección|dirección|notificación|fecha|radicado|asunto")
RE_INST   = re.compile(r"(?i)república|invima|ministerio|instituto|dirección|subdirección")

CSV_COLUMNS = [
    "Fecha de notificación",
    "Número de la alerta",
    "Nombre del producto",
    "RISARH",
    "Dirección de enlace",
]

# ============================================================
# SCRAPER — peticiones HTTP
# ============================================================

def get_page_html(page_index: int) -> str:
    url = BASE_URL if page_index == 0 else f"{BASE_URL}?page={page_index}"
    response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.text


def find_pdf_links(html: str) -> list:
    soup = BeautifulSoup(html, "lxml")
    links = [
        urljoin(BASE_URL, a.get("href"))
        for a in soup.find_all("a")
        if (a.get_text() or "").strip().lower() == "ver" and a.get("href")
    ]
    pdf_links = [u for u in links if ".pdf" in u.lower()]
    return list(dict.fromkeys(pdf_links))  # elimina duplicados


def fetch_pdf_bytes(url: str, session: requests.Session) -> bytes:
    for attempt in range(1, MAX_RETRIES + 1):
        response = session.get(url, headers=HEADERS, timeout=PDF_TIMEOUT)
        if response.status_code == 200 and response.content[:4] == b'%PDF':
            return response.content
        print(f"    Intento {attempt}/{MAX_RETRIES} — HTTP {response.status_code}")
        if attempt < MAX_RETRIES:
            time.sleep(RETRY_WAIT * attempt)
    raise Exception(f"HTTP {response.status_code} tras {MAX_RETRIES} intentos: {url}")

# ============================================================
# PARSER — extracción de datos desde PDF
# ============================================================

def _empty_record(pdf_url: str) -> dict:
    return {col: "" for col in CSV_COLUMNS} | {"Dirección de enlace": pdf_url}


def _extract_alert_number(text: str) -> str:
    match = RE_ALERT.search(text)
    if not match:
        return ""
    tipo   = re.sub(r"\s+", " ", match.group(1)).strip()  # "Alerta" o "Informe de Seguridad"
    numero = match.group(2).strip()
    return f"{tipo} No. {numero}".replace("No. No.", "No.")


def _extract_date(text: str) -> str:
    match = RE_DATE.search(text)
    if not match:
        return ""
    cleaned = re.sub(r"Bogotá\s*", "Bogotá", match.group(0), flags=re.IGNORECASE)
    return re.sub(r"\s{2,}", " ", cleaned).strip()


RE_NOMBRE = re.compile(r"(?i)^Nombre\s+del\s+producto\s*:\s*(.+)$")
RE_SKIP_TITLE = re.compile(r"(?i)^(invima\s+alerta|alerta\s+sanitaria|dirección\s+de)$")

def _extract_product_name(lines: list, date_match) -> str:
    # Estrategia 1: campo explícito "Nombre del producto: ..."
    for line in lines:
        m = RE_NOMBRE.match(line)
        if m and m.group(1).strip():
            return m.group(1).strip()

    # Estrategia 2: tras la fecha, bloque de líneas en mayúsculas (título largo)
    if date_match:
        date_norm = date_match.group(0).lower()
        date_idx = next((i for i, ln in enumerate(lines) if date_norm in ln.lower()), None)
        if date_idx is not None:
            title_parts = []
            for line in lines[date_idx + 1: date_idx + 15]:
                if RE_SKIP.search(line) or RE_SKIP_TITLE.match(line):
                    continue
                if line.isupper() and len(line) >= 6:
                    title_parts.append(line)
                elif title_parts:
                    break  # fin del bloque en mayúsculas
                elif len(line) >= 6:
                    return line  # primera línea mixta válida
            if title_parts:
                return " ".join(title_parts)

    # Fallback: primeras líneas con forma de título
    for line in lines[:20]:
        if (line.isupper() or line == line.title()) and 5 < len(line) < 160:
            if not RE_INST.search(line) and not RE_SKIP_TITLE.match(line):
                return line
    return ""


def extract_from_pdf(pdf_bytes: bytes, pdf_url: str) -> dict:
    record = _empty_record(pdf_url)
    try:
        reader = PdfReader(io.BytesIO(pdf_bytes))
        if not reader.pages:
            return record
        text  = (reader.pages[0].extract_text() or "").strip()
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        date_match = RE_DATE.search(text)

        record["Número de la alerta"]   = _extract_alert_number(text)
        record["Fecha de notificación"] = _extract_date(text)
        record["Nombre del producto"]   = _extract_product_name(lines, date_match)
        record["RISARH"]                = (m := RE_RISARH.search(text)) and m.group(0).upper() or ""
    except Exception:
        pass
    return record

# ============================================================
# RUNNER — orquesta el flujo
# ============================================================

def scrape(n_pages: int = 1) -> list:
    records, seen = [], set()
    session = requests.Session()
    session.get(BASE_URL, headers=HEADERS, timeout=REQUEST_TIMEOUT)  # establece cookies
    for page in range(n_pages):
        print(f"📄 Procesando página {page + 1} de {n_pages}...")
        html = get_page_html(page)
        for url in find_pdf_links(html):
            if url in seen:
                continue
            seen.add(url)
            try:
                record = extract_from_pdf(fetch_pdf_bytes(url, session), url)
            except Exception as e:
                print(f"  ⚠️  Error en {url}:\n      {e}")
                record = _empty_record(url)
            records.append(record)
            time.sleep(PAUSE_BETWEEN_PDFS)
        time.sleep(PAUSE_BETWEEN_PAGES)
    return records

# ============================================================
# EXPORTADOR — guarda el CSV
# ============================================================

def export_to_csv(records: list, filename: str = None) -> str:
    if not records:
        print("No se extrajo información.")
        return ""
    filepath = filename or f"alertas_invima_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    pd.DataFrame.from_records(records, columns=CSV_COLUMNS).to_csv(
        filepath, index=False, encoding="utf-8-sig"
    )
    print(f"✅ CSV guardado: {filepath}  ({len(records)} registros)")
    return filepath

# ============================================================
# EJECUCIÓN
# ============================================================

try:
    n_pages = int(input("¿Cuántas páginas deseas inspeccionar? (ej. 3): ").strip())
except ValueError:
    print("Entrada inválida. Usando 1 página.")
    n_pages = 1

records = scrape(n_pages=n_pages)
export_to_csv(records)

¿Cuántas páginas deseas inspeccionar? (ej. 3): 1
📄 Procesando página 1 de 1...
✅ CSV guardado: alertas_invima_20260605_132647.csv  (8 registros)


'alertas_invima_20260605_132647.csv'

In [1]:
# ============================================================
# INVIMA - Scraper de alertas de Dispositivos Médicos
# ============================================================
#   1. Ejecuta esta celda de instalación primero
# ============================================================

!pip install -q requests beautifulsoup4 lxml pymupdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 35.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# INVIMA - Scraper de alertas de Dispositivos Médicos y Otras Tecnologías
# ============================================================
#   2. Luego ejecuta este script
# ============================================================

import re
import time
from datetime import datetime
from urllib.parse import urljoin
import requests
import fitz  # PyMuPDF
import pandas as pd
from bs4 import BeautifulSoup

# -------------------------------------------------
# Configuración
# -------------------------------------------------
BASE_URL = "https://app.invima.gov.co/alertas/dispositivos-medicos-invima"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) Chrome/120 Safari/537.36",
    "Accept-Language": "es-CO,es;q=0.9",
    "Referer": BASE_URL,
}
TIMEOUT = 60

# -------------------------------------------------
# Patrones de extracción (PDF)
# -------------------------------------------------
SPANISH_MONTHS = r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|setiembre|octubre|noviembre|diciembre)"

# Fecha textual tipo: "Bogotá, 11 noviembre 2025" (acepta "Bogotá D.C." y "de")
RE_DATE = re.compile(
    r"(?i)\bBogot(?:á|a)(?:\s+D\.C\.)?,?\s*\d{1,2}\s*(?:de\s*)?"
    + SPANISH_MONTHS
    + r"\s*(?:de\s*)?\d{4}\b"
)

# Número de alerta / informe
RE_ALERTNUM = re.compile(r"(?i)\b(?:Alerta|Informe de Seguridad)\s*No\.?\s*[\w\-–—/]+")
# Asunto y Nombre del producto
RE_ASUNTO = re.compile(r"(?is)\bAsunto\s*:\s*(.+?)\s*(?:\n|\r|$)")
RE_NOMBRE_PRODUCTO = re.compile(r"(?is)\bNombre del producto\s*:\s*(.+?)\s*(?:\n|\r|$)")
# Código RISARH (heurístico)
RE_RISARH = re.compile(r"\b(?:MA|DA|DR|DI|RDI|RDR)\d{3,}-\d{2,}\b", re.IGNORECASE)

# Mapa de meses (para normalización desde fecha numérica)
MONTH_ES = {
    1: "enero", 2: "febrero", 3: "marzo", 4: "abril", 5: "mayo", 6: "junio",
    7: "julio", 8: "agosto", 9: "septiembre", 10: "octubre",
    11: "noviembre", 12: "diciembre",
}

# -------------------------------------------------
# Helpers HTTP/HTML
# -------------------------------------------------
def get_page_html(page_index: int) -> str:
    """Descarga HTML de la lista (manejo de paginación ?page=N)."""
    url = BASE_URL if page_index == 0 else f"{BASE_URL}?page={page_index}"
    r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text

def find_detail_links_in_page(html: str, page_url: str) -> list:
    """
    Devuelve hrefs (absolutos) de la columna 'Documento' (botón 'Ver').
    Estos hrefs pueden ser PDFs directos o páginas de detalle.
    """
    soup = BeautifulSoup(html, "lxml")
    links = []
    for a in soup.find_all("a"):
        txt = (a.get_text() or "").strip().lower()
        href = (a.get("href") or "").strip()
        if txt == "ver" and href:
            abs_href = urljoin(page_url, href)
            links.append(abs_href)
    seen, uniq = set(), []
    for u in links:
        if u not in seen:
            uniq.append(u)
            seen.add(u)
    return uniq

def resolve_pdf_link(href: str) -> str:
    """
    Si href ya es un PDF, lo retorna. Si es página de detalle, intenta encontrar un PDF.
    """
    if ".pdf" in href.lower():
        return href
    try:
        r = requests.get(href, headers={**HEADERS, "Referer": href}, timeout=TIMEOUT)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        for a in soup.find_all("a", href=True):
            if ".pdf" in a["href"].lower():
                return urljoin(href, a["href"])
        for a in soup.find_all("a", href=True):
            t = (a.get_text() or "").strip().lower()
            if "ver alerta" in t or "descargar" in t or "ver documento" in t:
                return urljoin(href, a["href"])
    except Exception:
        pass
    return href

# -------------------------------------------------
# PDF: Descarga y extracción de campos
# -------------------------------------------------
def fetch_pdf_bytes(url: str) -> bytes:
    r = requests.get(url, headers=HEADERS, timeout=max(90, TIMEOUT))
    r.raise_for_status()
    return r.content

def _first_page_text(pdf_bytes: bytes) -> str:
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        if doc.page_count == 0:
            return ""
        return (doc[0].get_text("text") or "").strip()

def _clean_for_date(text: str) -> str:
    """
    Normaliza el texto para facilitar el parseo de fechas:
    - Colapsa espacios, elimina caracteres invisibles y NBSP.
    - Homologa 'Bogota'/'Bogotá'.
    - Sustituye separadores de fecha variados por '/'.
    - Quita puntuación final (; . ,) a la derecha de la fecha.
    """
    t = text
    t = t.replace("\u00A0", " ").replace("\u200b", "")
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"(?i)\bBogota\b", "Bogotá", t)
    t = re.sub(r"([0-9])\s*[-–—\.]\s*([0-9])", r"\1/\2", t)
    t = re.sub(r"(?i)\bBogotá(?:\s+D\.C\.)?,?\s*", "Bogotá, ", t)
    return t

def _extract_fecha(text: str) -> str:
    """
    Extrae y normaliza la fecha de notificación en formato:
    'Bogotá, DD mes YYYY'
    Acepta también variantes numéricas próximas a 'Bogotá' como 'YYYY/MM/DD', 'DD/MM/YYYY',
    con separadores -, ., / y puntuación final.
    Corrige específicamente '2525' -> '2025'.
    """
    t = _clean_for_date(text)

    # 1) Intento con el patrón textual tradicional
    m_txt = RE_DATE.search(t)
    if m_txt:
        s = m_txt.group(0)
        s = re.sub(r"(?i)\bBogot(?:á|a)(?:\s+D\.C\.)?,?\s*", "Bogotá, ", s).strip()
        s = re.sub(r"(?i)\bsetiembre\b", "septiembre", s)
        s = re.sub(r"[;.,]\s*$", "", s)
        s = re.sub(r"\s{2,}", " ", s)
        return s

    # 2) Búsqueda robusta de tres números cercanos a 'Bogotá'
    m_ctx = re.search(r"(?i)Bogotá,\s*([0-9][^A-Za-z]{0,20})", t)
    candidate = None
    if m_ctx:
        tail = m_ctx.group(1)
        m_nums = re.search(r"(\d{1,4})\D+(\d{1,2})\D+(\d{1,4})", tail)
        if m_nums:
            a, b, c = m_nums.group(1), m_nums.group(2), m_nums.group(3)
            year = month = day = None
            if len(a) == 4:
                year, month, day = int(a), int(b), int(c)
            elif len(c) == 4:
                day, month, year = int(a), int(b), int(c)

            if year is not None and month is not None and day is not None:
                if year == 2525:
                    year = 2025
                if 1 <= month <= 12 and 1 <= day <= 31:
                    mes_txt = MONTH_ES.get(month, "")
                    return f"Bogotá, {day} {mes_txt} {year}"

    # 3) Último intento: buscar en todo el texto, sin anclar a 'Bogotá'
    m_nums2 = re.search(r"(\d{4})\D+(\d{1,2})\D+(\d{1,2})[;.,]?", t)
    if m_nums2:
        year, month, day = int(m_nums2.group(1)), int(m_nums2.group(2)), int(m_nums2.group(3))
        if year == 2525:
            year = 2025
        if 1 <= month <= 12 and 1 <= day <= 31:
            mes_txt = MONTH_ES.get(month, "")
            return f"Bogotá, {day} {mes_txt} {year}"

    return ""

def _extract_numero_alerta(text: str) -> str:
    m = RE_ALERTNUM.search(text)
    return m.group(0).strip().replace("  ", " ") if m else ""

def _extract_nombre_producto(text: str, lines: list, fecha_match: str) -> str:
    m_asunto = RE_ASUNTO.search(text)
    if m_asunto:
        return re.sub(r"\s+", " ", m_asunto.group(1)).strip(" :–—\u200b")
    m_nombre = RE_NOMBRE_PRODUCTO.search(text)
    if m_nombre:
        return re.sub(r"\s+", " ", m_nombre.group(1)).strip(" :–—\u200b")
    if fecha_match:
        fecha_idx = None
        fecha_norm = fecha_match.lower()
        for i, ln in enumerate(lines):
            if fecha_norm in ln.lower():
                fecha_idx = i
                break
        if fecha_idx is not None:
            for j in range(fecha_idx + 1, min(fecha_idx + 12, len(lines))):
                cand = lines[j].strip()
                if len(cand) < 6:
                    continue
                if re.search(r"(?i)\b(república|invima|ministerio|dirección|subdirección|radicado|notificación|fecha|referencia)\b", cand):
                    continue
                return cand
    for ln in lines[:15]:
        tline = ln.strip()
        if 6 < len(tline) < 180 and (tline.isupper() or tline == tline.title()):
            if not re.search(r"(?i)\b(república|invima|ministerio|dirección|subdirección)\b", tline):
                return tline
    return ""

def _extract_risarh(text: str) -> str:
    m = RE_RISARH.search(text)
    return m.group(0).upper() if m else ""

def extract_from_pdf_bytes(pdf_bytes: bytes, pdf_url: str) -> dict:
    out = {
        "Fecha de notificación": "",
        "Número de la alerta": "",
        "Nombre del producto": "",
        "RISARH": "",
        "Estado": "",
        "Dirección de enlace": pdf_url,
    }
    try:
        text = _first_page_text(pdf_bytes)
        if not text:
            return out
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        out["Número de la alerta"] = _extract_numero_alerta(text)
        fecha = _extract_fecha(text)
        out["Fecha de notificación"] = fecha
        out["Nombre del producto"] = _extract_nombre_producto(text, lines, fecha)
        out["RISARH"] = _extract_risarh(text)
        out["Estado"] = ""
    except Exception:
        pass
    return out

# -------------------------------------------------
# Scraper principal
# -------------------------------------------------
def scrape_invima(n_pages: int = 1, pause: float = 0.5) -> list:
    results = []
    seen_links = set()
    for page in range(n_pages):
        page_url = BASE_URL if page == 0 else f"{BASE_URL}?page={page}"
        try:
            html = get_page_html(page)
        except Exception:
            time.sleep(pause)
            continue
        detail_links = find_detail_links_in_page(html, page_url)
        for link in detail_links:
            if link in seen_links:
                continue
            seen_links.add(link)
            pdf_url = resolve_pdf_link(link)
            try:
                if ".pdf" not in pdf_url.lower():
                    results.append({
                        "Fecha de notificación": "",
                        "Número de la alerta": "",
                        "Nombre del producto": "",
                        "RISARH": "",
                        "Estado": "",
                        "Dirección de enlace": pdf_url,
                    })
                    continue
                pdf_bytes = fetch_pdf_bytes(pdf_url)
                record = extract_from_pdf_bytes(pdf_bytes, pdf_url)
                results.append(record)
                time.sleep(0.1)
            except Exception:
                results.append({
                    "Fecha de notificación": "",
                    "Número de la alerta": "",
                    "Nombre del producto": "",
                    "RISARH": "",
                    "Estado": "",
                    "Dirección de enlace": pdf_url,
                })
        time.sleep(pause)
    return results

# -------------------------------------------------
# Exportación (CSV solamente)
# -------------------------------------------------
def export_records(records: list, base_filename: str | None = None):
    if not records:
        print("No se extrajo información.")
        return None
    df = pd.DataFrame.from_records(records, columns=[
        "Fecha de notificación",
        "Número de la alerta",
        "Nombre del producto",
        "RISARH",
        "Estado",
        "Dirección de enlace",
    ])
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_filename = base_filename or f"alertas_invima_dm_{ts}"
    csv_path = f"{base_filename}.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"✅ CSV: {csv_path}")
    return csv_path

# -------------------------------------------------
# Ejecución
# -------------------------------------------------
try:
    n_pages = int(input("¿Cuántas páginas deseas inspeccionar? (ej. 3): ").strip())
except Exception:
    print("Entrada inválida. Usaré 1 página.")
    n_pages = 1

print(f"Scraping de {n_pages} página(s)...")
records = scrape_invima(n_pages=n_pages, pause=0.6)
export_records(records)

¿Cuántas páginas deseas inspeccionar? (ej. 3): 3
Scraping de 3 página(s)...
✅ CSV: alertas_invima_dm_20260612_170133.csv


'alertas_invima_dm_20260612_170133.csv'